Carlos Giral Herrero
## Técnicas de Optimización No Lineal y Estocástica. PROBLEMA 2

En este trabajo se aborda un problema de planificación financiera mediante optimización estocástica multietapa. Se parte de una riqueza inicial 
$b = 55\,000\$ $ y se desea alcanzar, al final del horizonte, una riqueza objetivo $W = 80\,000\$ $. El horizonte temporal es de 15 años y se toman decisiones de inversión cada 5 años, por lo que el problema se modeliza con $T = 4$ etapas (decisión inicial y tres realizaciones sucesivas de incertidumbre).

En cada etapa se decide la cantidad a invertir en dos productos financieros $I = \{\text{activos}, \text{bonos}\}$. Los rendimientos de dichos productos son inciertos y se representan mediante un árbol de escenarios: en cada nodo la pareja de rendimientos $(\xi_1, \xi_2)$ puede tomar los valores $(1.25,\,1.14)$ con probabilidad $p$ o $(1.06,\,1.12)$ con probabilidad $1-p$, donde 
$$
p = \frac{1}{N+2},
$$
siendo $N$ el último dígito del DNI (En mi caso: $N=9$).

El modelo neutral al riesgo maximiza la utilidad esperada del excedente o déficit respecto al objetivo final $W$. En la etapa final, el excedente $y$ sobre $W$ se recomensa a un interés $q^+ = 1\%$, mientras que el déficit $w$ se penaliza a un interés $q^- = 4\%$. Además, las decisiones deben respetar la no anticipatividad, es decir, en nodos con la misma información disponible hasta una etapa dada, las decisiones deben coindidir.

## (c) Plantea el problema estocástico multietapa en formulación compacta. Resuélvelo e indica el valor de la función objetivo y la solución. Interpreta la solución.

## 1. Conjuntos, parámetros y variables

### Conjuntos
- Productos financieros: $I = \{1,2\}$, donde  
  $i=1$ = *activos*, $i=2$ = *bonos*.
- Etapas de decisión: $t = 1,2,3,4$ (con $T=4$).
- Nodos del árbol de escenarios: $g \in G_t$ en cada etapa $t$.
- Hojas (nodos finales): $g \in G_T$.

Se denota por $\pi(g)$ el **nodo padre** inmediato de $g$ (predecesor en el árbol).

---

### Parámetros
- Riqueza inicial: $b = 55\,000\$ $.
- Objetivo de riqueza final: $W = 80\,000\$ $.
- Intereses sobre excedente y déficit (en tanto por uno):  
  $q^+ = 0.01$, $q^- = 0.04$.
- Probabilidad del escenario "alto":  
  $p = \frac{1}{N+2} = \frac{1}{11}$ (con $N=9$).  
  Por tanto, $1-p = \frac{10}{11}$.
- Rendimientos por etapa y nodo:  
  Para cada nodo $g$ (excepto el raíz) y producto $i\in I$, el rendimiento es $\xi_i^g$, tomando valores:
  - High: $(\xi_1^g,\xi_2^g) = (1.25,\,1.14)$ con probabilidad $p$
  - Low:  $(\xi_1^g,\xi_2^g) = (1.06,\,1.12)$ con probabilidad $1-p$

- Probabilidad de cada hoja $g \in G_T$: $p_g$ (producto de probabilidades a lo largo del camino desde la raíz).

---

### Variables de decisión
- $x_i^g \ge 0$: cantidad invertida en el producto $i\in I$ en el nodo $g$ (para $g \in G_t$, $t < T$).
- $y^g \ge 0$: excedente final sobre el objetivo $W$ en la hoja $g \in G_T$.
- $w^g \ge 0$: déficit final respecto al objetivo $W$ en la hoja $g \in G_T$.

## 2. Formulación compacta multietapa (modelo neutral al riesgo)

El modelo neutral al riesgo es:

$$
Z_{RN}=\max \sum_{g\in G_T} p_g\,(q^+\,y^g - q^-\,w^g)
$$

sujeto a:

**(i) Presupuesto inicial (raíz)**

$$
\sum_{i\in I} x_i^{1} = b.
$$

**(ii) Balance en nodos intermedios (reinversión total)**  
Para todo nodo $g\in G_t$, con $t=2,\dots,T-1$:

$$
\sum_{i\in I} x_i^{g} - \sum_{i\in I}\xi_i^{g}\,x_i^{\pi(g)} = 0.
$$

**(iii) Condición final respecto al objetivo $W$**  
Para toda hoja $g\in G_T$:

$$
\sum_{i\in I}\xi_i^{g}\,x_i^{\pi(g)} - y^g + w^g = W.
$$

**(iv) No negatividad**

$$
x_i^g \ge 0,\ \forall i\in I,\ g\in G_t,\ t<T;\qquad
y^g\ge 0,\ w^g\ge 0,\ \forall g\in G_T.
$$

## 3. Resolución y valor de la función objetivo

In [1]:
import numpy as np
import pandas as pd
from scipy.optimize import linprog

# -------------------------
# Parámetros del problema
# -------------------------
b = 55_000.0
W = 80_000.0
q_plus = 0.01
q_minus = 0.04

N = 9
p = 1/(N+2)
p_low = 1 - p

R_high = (1.25, 1.14)  # (activos, bonos)
R_low  = (1.06, 1.12)

# -------------------------
# Árbol de escenarios (T=4)
# Nodos como en las diapositivas: 1 (raíz), 2-3, 4-7, 8-15 (hojas)
# Convención: hijo "izq" = High, hijo "der" = Low
# -------------------------
stages = {
    1: [1],
    2: [2, 3],
    3: [4, 5, 6, 7],
    4: [8, 9, 10, 11, 12, 13, 14, 15],
}
parent = {2:1, 3:1, 4:2, 5:2, 6:3, 7:3, 8:4, 9:4, 10:5, 11:5, 12:6, 13:6, 14:7, 15:7}

# realización (rendimiento) asociada a la transición hacia el nodo g (g != 1)
real = {2:'H', 3:'L', 4:'H', 5:'L', 6:'H', 7:'L', 8:'H', 9:'L', 10:'H', 11:'L', 12:'H', 13:'L', 14:'H', 15:'L'}

xi = {g: (R_high if real[g]=='H' else R_low) for g in range(2, 16)}

def leaf_prob(g):
    prob = 1.0
    cur = g
    while cur != 1:
        prob *= (p if real[cur]=='H' else p_low)
        cur = parent[cur]
    return prob

leaves = stages[4]
p_leaf = {g: leaf_prob(g) for g in leaves}

# -------------------------
# Variables:
# x_i^g para g en {1..7} y i=1,2
# y^g, w^g para hojas g en {8..15}
# -------------------------
nodes_x = [1,2,3,4,5,6,7]

idx = {}
k = 0
for g in nodes_x:
    for i in [1,2]:
        idx[('x', g, i)] = k; k += 1
for g in leaves:
    idx[('y', g)] = k; k += 1
for g in leaves:
    idx[('w', g)] = k; k += 1
n = k

# Objetivo: max sum p_g (q+ y - q- w)  <=>  min c^T var
c = np.zeros(n)
for g in leaves:
    c[idx[('y', g)]] = -p_leaf[g] * q_plus
    c[idx[('w', g)]] =  p_leaf[g] * q_minus

A_eq = []
b_eq = []

# (i) Presupuesto inicial
row = np.zeros(n)
row[idx[('x', 1, 1)]] = 1
row[idx[('x', 1, 2)]] = 1
A_eq.append(row); b_eq.append(b)

# (ii) Balance nodos intermedios t=2..T-1 -> aquí g=2..7
for g in [2,3,4,5,6,7]:
    row = np.zeros(n)
    row[idx[('x', g, 1)]] = 1
    row[idx[('x', g, 2)]] = 1
    pg = parent[g]
    row[idx[('x', pg, 1)]] -= xi[g][0]
    row[idx[('x', pg, 2)]] -= xi[g][1]
    A_eq.append(row); b_eq.append(0.0)

# (iii) Condición final en hojas
for g in leaves:
    row = np.zeros(n)
    pg = parent[g]
    row[idx[('x', pg, 1)]] = xi[g][0]
    row[idx[('x', pg, 2)]] = xi[g][1]
    row[idx[('y', g)]] = -1
    row[idx[('w', g)]] =  1
    A_eq.append(row); b_eq.append(W)

A_eq = np.array(A_eq)
b_eq = np.array(b_eq)

bounds = [(0, None)] * n

res = linprog(c, A_eq=A_eq, b_eq=b_eq, bounds=bounds, method="highs")
res.status, res.message

(0, 'Optimization terminated successfully. (HiGHS Status 7: Optimal)')

In [2]:
x = res.x
Z_RN = -res.fun  # porque minimizamos el negativo

def get_x(g,i): return x[idx[('x', g, i)]]
def get_y(g):   return x[idx[('y', g)]]
def get_w(g):   return x[idx[('w', g)]]

# Tabla decisiones x
sol_x = pd.DataFrame([{
    "node": g,
    "stage": next(t for t,gs in stages.items() if g in gs),
    "x_activos": get_x(g,1),
    "x_bonos": get_x(g,2),
} for g in nodes_x])

# Tabla hojas (stage 4): y,w + riqueza final + desglose por producto
rows = []
for g in leaves:
    pg = parent[g]  # nodo padre en stage 3

    x_act_parent = get_x(pg, 1)
    x_bon_parent = get_x(pg, 2)

    wealth_act = xi[g][0] * x_act_parent
    wealth_bon = xi[g][1] * x_bon_parent
    wealth = wealth_act + wealth_bon

    rows.append({
        "leaf": g,
        "stage": 4,
        "prob": p_leaf[g],
        "parent_node": pg,
        "x_act_parent": x_act_parent,
        "x_bon_parent": x_bon_parent,
        "wealth_activos": wealth_act,
        "wealth_bonos": wealth_bon,
        "wealth_final": wealth,
        "y_exceso": get_y(g),
        "w_deficit": get_w(g),
        "%bonos": (wealth_bon / wealth * 100.0) if wealth > 1e-12 else 0.0
    })

sol_yw = pd.DataFrame(rows)

print("===== Resultados (c) modelo neutral al riesgo =====")
print(f"Solver status: {res.status} | message: {res.message}")
print(f"Z_RN (máximo) = {Z_RN:.4f}")
display(sol_x.round(4))
display(sol_yw.round(4))

===== Resultados (c) modelo neutral al riesgo =====
Solver status: 0 | message: Optimization terminated successfully. (HiGHS Status 7: Optimal)
Z_RN (máximo) = -94.1521


,node,stage,x_activos,x_bonos
0,1,1,0.0,55000.0
1,2,2,0.0,62700.0
2,3,2,0.0,61600.0
3,4,3,0.0,71478.0
4,5,3,0.0,70224.0
5,6,3,0.0,70224.0
6,7,3,0.0,68992.0


,leaf,stage,prob,parent_node,x_act_parent,x_bon_parent,wealth_activos,wealth_bonos,wealth_final,y_exceso,w_deficit,%bonos
0,8,4,0.0008,4,0.0,71478.0,0.0,81484.92,81484.92,1484.92,0.00,100.0
1,9,4,0.0075,4,0.0,71478.0,0.0,80055.36,80055.36,55.36,0.00,100.0
2,10,4,0.0075,5,0.0,70224.0,0.0,80055.36,80055.36,55.36,0.00,100.0
3,11,4,0.0751,5,0.0,70224.0,0.0,78650.88,78650.88,0.00,1349.12,100.0
4,12,4,0.0075,6,0.0,70224.0,0.0,80055.36,80055.36,55.36,0.00,100.0
5,13,4,0.0751,6,0.0,70224.0,0.0,78650.88,78650.88,0.00,1349.12,100.0
6,14,4,0.0751,7,0.0,68992.0,0.0,78650.88,78650.88,0.00,1349.12,100.0
7,15,4,0.7513,7,0.0,68992.0,0.0,77271.04,77271.04,0.00,2728.96,100.0


## 4. Interpretación de resultados (modelo neutral al riesgo)

La solución óptima concentra la inversión en **bonos** en todas las etapas y nodos del árbol. Esto se explica por dos motivos principales:

1. **Asimetría premio–castigo**: la penalización del déficit es mayor que la recompensa del excedente ($q^- = 4\%$ frente a $q^+ = 1\%$), por lo que el modelo prioriza evitar quedar por debajo de $W$.
2. **Alta probabilidad del escenario desfavorable**: en este caso $N=9$, luego
   $$
   p=\frac{1}{N+2}=\frac{1}{11}\approx 0.0909,\qquad 1-p=\frac{10}{11}\approx 0.9091.
   $$
   Por tanto, las trayectorias con realizaciones *Low* son muy probables y dominan la esperanza. Además, en *Low* el bono rinde más que el activo ($1.12$ frente a $1.06$), lo que refuerza una política conservadora basada en bonos.

En consecuencia, el modelo puede generar excedente en algunos escenarios favorables, pero presenta déficit en escenarios muy desfavorables (secuencias con varios *Low*), lo cual se refleja en $w^g>0$ en varias hojas y en un valor esperado de utilidad final $Z_{RN}$ negativo.


## (d) Plantea el problema estocástico que optimiza el peor escenario. Resuélvelo e indica el valor de la función objetivo y la solución bajo cada escenario. Interpreta los resultados

## 1. Formulación compacta multietapa (criterio de peor escenario / minimax)

El problema de peor escenario (minimax) maximiza el valor garantizado de la utilidad final. Para ello se introduce una variable auxiliar $V$ (cota inferior común de la utilidad en las hojas).

$$
Z_{\text{WC}}=\max \; V
$$

sujeto a:

**(i) Presupuesto inicial (raíz)**

$$
\sum_{i\in I} x_i^{1} = b.
$$

**(ii) Balance en nodos intermedios (reinversión total)**  
Para todo nodo $g\in G_t$, con $t=2,\dots,T-1$:

$$
\sum_{i\in I} x_i^{g} - \sum_{i\in I}\xi_i^{g}\,x_i^{\pi(g)} = 0.
$$

**(iii) Condición final respecto al objetivo $W$**  
Para toda hoja $g\in G_T$:

$$
\sum_{i\in I}\xi_i^{g}\,x_i^{\pi(g)} - y^g + w^g = W.
$$

**(iv) Restricciones de peor escenario (minimax)**  
Para toda hoja $g\in G_T$:

$$
V \le q^+\,y^g - q^-\,w^g.
$$

**(v) No negatividad**

$$
x_i^g \ge 0,\ \forall i\in I,\ g\in G_t,\ t<T;\qquad
y^g\ge 0,\ w^g\ge 0,\ \forall g\in G_T;\qquad
V\in\mathbb{R}.
$$

## 2. Resolución y valor de la función objetivo

In [3]:
# =========================
# (d) Peor escenario (minimax)
# =========================

# 1) Rehacer el índice incluyendo la variable V (al final)
idx_d = {}
k = 0
for g in nodes_x:
    for i in [1,2]:
        idx_d[('x', g, i)] = k; k += 1
for g in leaves:
    idx_d[('y', g)] = k; k += 1
for g in leaves:
    idx_d[('w', g)] = k; k += 1
idx_d[('V',)] = k; k += 1
n_d = k

# 2) Objetivo: max V  <=> min (-V)
c_d = np.zeros(n_d)
c_d[idx_d[('V',)]] = -1.0

# 3) Igualdades
A_eq_d = []
b_eq_d = []

# Presupuesto inicial
row = np.zeros(n_d)
row[idx_d[('x', 1, 1)]] = 1
row[idx_d[('x', 1, 2)]] = 1
A_eq_d.append(row); b_eq_d.append(b)

# Balance nodos intermedios (g=2..7)
for g in [2,3,4,5,6,7]:
    row = np.zeros(n_d)
    row[idx_d[('x', g, 1)]] = 1
    row[idx_d[('x', g, 2)]] = 1
    pg = parent[g]
    row[idx_d[('x', pg, 1)]] -= xi[g][0]
    row[idx_d[('x', pg, 2)]] -= xi[g][1]
    A_eq_d.append(row); b_eq_d.append(0.0)

# Condición final (hojas)
for g in leaves:
    row = np.zeros(n_d)
    pg = parent[g]
    row[idx_d[('x', pg, 1)]] = xi[g][0]
    row[idx_d[('x', pg, 2)]] = xi[g][1]
    row[idx_d[('y', g)]] = -1
    row[idx_d[('w', g)]] =  1
    A_eq_d.append(row); b_eq_d.append(W)

A_eq_d = np.array(A_eq_d)
b_eq_d = np.array(b_eq_d)

# 4) Desigualdades: V <= q+ y^g - q- w^g
A_ub_d = []
b_ub_d = []
for g in leaves:
    row = np.zeros(n_d)
    row[idx_d[('V',)]] = 1
    row[idx_d[('y', g)]] = -q_plus
    row[idx_d[('w', g)]] =  q_minus
    A_ub_d.append(row); b_ub_d.append(0.0)

A_ub_d = np.array(A_ub_d)
b_ub_d = np.array(b_ub_d)

# 5) Cotas: x,y,w >=0 y V libre
bounds_d = [(0, None)]*(n_d-1) + [(None, None)]

res_d = linprog(
    c_d,
    A_ub=A_ub_d, b_ub=b_ub_d,
    A_eq=A_eq_d, b_eq=b_eq_d,
    bounds=bounds_d,
    method="highs"
)

x_d = res_d.x
V_star = x_d[idx_d[('V',)]]

In [4]:
def get_xd(g,i): return x_d[idx_d[('x', g, i)]]
def get_yd(g):   return x_d[idx_d[('y', g)]]
def get_wd(g):   return x_d[idx_d[('w', g)]]

# --- Resumen principal ---
print("===== Resultados (d) Peor escenario (minimax) =====")
print(f"Solver status: {res_d.status} | message: {res_d.message}")
print(f"Z_WC = V* (máximo) = {V_star:.4f}")

# Tabla de decisiones
sol_x_d = pd.DataFrame([{
    "node": g,
    "stage": next(t for t,gs in stages.items() if g in gs),
    "x_activos": get_xd(g,1),
    "x_bonos": get_xd(g,2),
} for g in nodes_x])

# Tabla por escenario final 
rows = []
for g in leaves:
    pg = parent[g]  # nodo padre (stage 3)

    x_act_parent = get_xd(pg, 1)
    x_bon_parent = get_xd(pg, 2)

    wealth_act = xi[g][0] * x_act_parent
    wealth_bon = xi[g][1] * x_bon_parent
    wealth = wealth_act + wealth_bon

    y = get_yd(g)
    w = get_wd(g)
    u = q_plus*y - q_minus*w

    rows.append({
        "leaf": g,
        "stage": 4,
        "prob": p_leaf[g],          # informativa (no influye en minimax)
        "parent_node": pg,
        "x_act_parent": x_act_parent,
        "x_bon_parent": x_bon_parent,
        "wealth_activos": wealth_act,
        "wealth_bonos": wealth_bon,
        "wealth_final": wealth,
        "y_exceso": y,
        "w_deficit": w,
        "%bonos": (wealth_bon/wealth*100.0) if wealth > 1e-12 else 0.0,
        "u_g": u,
        "¿peor?": abs(u - V_star) < 1e-6
    })

sol_leaf_d = pd.DataFrame(rows).sort_values("leaf")

display(sol_x_d.round(4))
display(sol_leaf_d.round(4))

===== Resultados (d) Peor escenario (minimax) =====
Solver status: 0 | message: Optimization terminated successfully. (HiGHS Status 7: Optimal)
Z_WC = V* (máximo) = -109.1584


,node,stage,x_activos,x_bonos
0,1,1,0.0000,55000.0000
1,2,2,20533.3333,42166.6667
2,3,2,0.0000,61600.0000
3,4,3,73736.6667,0.0000
4,5,3,0.0000,68992.0000
5,6,3,22997.3333,47226.6667
6,7,3,0.0000,68992.0000


,leaf,stage,prob,parent_node,x_act_parent,x_bon_parent,wealth_activos,wealth_bonos,wealth_final,y_exceso,w_deficit,%bonos,u_g,¿peor?
0,8,4,0.0008,4,73736.6667,0.0000,92170.8333,0.0000,92170.8333,12170.8333,0.0000,0.0000,121.7083,False
1,9,4,0.0075,4,73736.6667,0.0000,78160.8667,0.0000,78160.8667,0.0000,1839.1333,0.0000,-73.5653,False
2,10,4,0.0075,5,0.0000,68992.0000,0.0000,78650.8800,78650.8800,0.0000,1349.1200,100.0000,-53.9648,False
3,11,4,0.0751,5,0.0000,68992.0000,0.0000,77271.0400,77271.0400,0.0000,2728.9600,100.0000,-109.1584,True
4,12,4,0.0075,6,22997.3333,47226.6667,28746.6667,53838.4000,82585.0667,2585.0667,0.0000,65.1914,25.8507,False
5,13,4,0.0751,6,22997.3333,47226.6667,24377.1733,52893.8667,77271.0400,0.0000,2728.9600,68.4524,-109.1584,True
6,14,4,0.0751,7,0.0000,68992.0000,0.0000,78650.8800,78650.8800,0.0000,1349.1200,100.0000,-53.9648,False
7,15,4,0.7513,7,0.0000,68992.0000,0.0000,77271.0400,77271.0400,0.0000,2728.9600,100.0000,-109.1584,True


### 3. Interpretación de resultados (criterio de peor escenario / minimax)

El modelo minimax maximiza el valor garantizado $V$, es decir, la utilidad final en el escenario más desfavorable. En la solución obtenida se alcanza:

$$
Z_{WC}=V^*=-109.1584.
$$

La tabla por escenarios muestra varias hojas con $u_g=q^+y^g-q^-w^g$ exactamente iguales a $V^*$ (marcadas como `¿peor?=True`). Dichas hojas determinan el óptimo: no es posible aumentar $V$ sin mejorar simultáneamente todas ellas.

En particular, en los escenarios más desfavorables se obtiene un déficit final $w=2728.9600$ (con $y=0$), y por tanto:

$$
u_g=-q^-w=-0.04\cdot 2728.9600=-109.1584.
$$

Esto indica que, incluso con un criterio de peor caso, no se puede garantizar alcanzar el objetivo $W=80\,000$ bajo el escenario final más adverso.

Finalmente, nótese que el valor óptimo de (c) ($Z_{RN}$) es una **esperanza** (media ponderada por probabilidades), mientras que en (d) $Z_{WC}$ es un **umbral garantizado** (peor caso), por lo que ambos valores no son comparables directamente.

## (e) Plantea el problema VaR que optimiza el primer quartil. Resuelve e indica el valor de la función objetivo y la solución bajo cada escenario. Interpreta los resultados.

## 1. Formulación compacta multietapa (VaR: primer cuartil)

En este apartado se optimiza el **primer cuartil (25%)** de la utilidad final. Definimos, para cada hoja $g\in G_T$, la utilidad:

$$
u_g = q^+\,y^g - q^-\,w^g.
$$

Se introduce una variable auxiliar $\eta\in\mathbb{R}$ que representa el **VaR del 25%** (primer cuartil) de dicha utilidad. El objetivo es maximizar $\eta$ imponiendo que, con probabilidad al menos $1-\alpha=0.75$, la utilidad sea mayor o igual que $\eta$.

El modelo se formula como:

$$
Z_{\text{VaR}}=\max \; \eta
$$

sujeto a:

**(i) Presupuesto inicial (raíz)**

$$
\sum_{i\in I} x_i^{1} = b.
$$

**(ii) Balance en nodos intermedios (reinversión total)**  
Para todo nodo $g\in G_t$, con $t=2,\dots,T-1$:

$$
\sum_{i\in I} x_i^{g} - \sum_{i\in I}\xi_i^{g}\,x_i^{\pi(g)} = 0.
$$

**(iii) Condición final respecto al objetivo $W$**  
Para toda hoja $g\in G_T$:

$$
\sum_{i\in I}\xi_i^{g}\,x_i^{\pi(g)} - y^g + w^g = W.
$$

**(iv) Restricciones VaR (primer cuartil)**  

Se selecciona un subconjunto de hojas $S\subseteq G_T$ tal que

$$
\sum_{g\in S} p_g \ge 0.75,
$$

y se exige:

$$
\eta \le q^+\,y^g - q^-\,w^g,\qquad \forall g\in S.
$$

Equivalente a imponer que la utilidad sea al menos $\eta$ en un conjunto de escenarios cuya probabilidad total sea al menos el $75\%$.

**(v) No negatividad**

$$
x_i^g \ge 0,\ \forall i\in I,\ g\in G_t,\ t<T;\qquad
y^g\ge 0,\ w^g\ge 0,\ \forall g\in G_T;\qquad
\eta\in\mathbb{R}.
$$

## 2. Resolución y valor de la función objetivo

In [5]:
import itertools
# =========================
# (e) VaR (primer cuartil) - Resolución por enumeración
# =========================

alpha = 0.25
target_prob = 1 - alpha  # 0.75

# 1) Índices (x,y,w,eta)
idx_e = {}
k = 0
for g in nodes_x:
    for i in [1, 2]:
        idx_e[('x', g, i)] = k; k += 1
for g in leaves:
    idx_e[('y', g)] = k; k += 1
for g in leaves:
    idx_e[('w', g)] = k; k += 1
idx_e[('eta',)] = k; k += 1
n_e = k

# 2) Igualdades (idénticas a (c) y (d))
A_eq_e = []
b_eq_e = []

# Presupuesto inicial
row = np.zeros(n_e)
row[idx_e[('x', 1, 1)]] = 1
row[idx_e[('x', 1, 2)]] = 1
A_eq_e.append(row); b_eq_e.append(b)

# Balance nodos intermedios (g=2..7)
for g in [2, 3, 4, 5, 6, 7]:
    row = np.zeros(n_e)
    row[idx_e[('x', g, 1)]] = 1
    row[idx_e[('x', g, 2)]] = 1
    pg = parent[g]
    row[idx_e[('x', pg, 1)]] -= xi[g][0]
    row[idx_e[('x', pg, 2)]] -= xi[g][1]
    A_eq_e.append(row); b_eq_e.append(0.0)

# Condición final (hojas)
for g in leaves:
    row = np.zeros(n_e)
    pg = parent[g]
    row[idx_e[('x', pg, 1)]] = xi[g][0]
    row[idx_e[('x', pg, 2)]] = xi[g][1]
    row[idx_e[('y', g)]] = -1
    row[idx_e[('w', g)]] =  1
    A_eq_e.append(row); b_eq_e.append(W)

A_eq_e = np.array(A_eq_e)
b_eq_e = np.array(b_eq_e)

# 3) Bounds: x,y,w >=0; eta libre
bounds_e = [(0, None)]*(n_e-1) + [(None, None)]

# 4) Objetivo: max eta <=> min -eta
c_e = np.zeros(n_e)
c_e[idx_e[('eta',)]] = -1.0

leaf_list = list(leaves)

best_eta = -np.inf
best_S = None
best_res = None

# Enumeración de subconjuntos S con prob>=0.75
for r in range(1, len(leaf_list)+1):
    for S in itertools.combinations(leaf_list, r):
        prob_S = sum(p_leaf[g] for g in S)
        if prob_S + 1e-12 < target_prob:
            continue

        # Restricciones VaR en S: eta <= q+ y_g - q- w_g
        # => eta - q+ y_g + q- w_g <= 0
        A_ub = []
        b_ub = []
        for g in S:
            row = np.zeros(n_e)
            row[idx_e[('eta',)]] = 1
            row[idx_e[('y', g)]] = -q_plus
            row[idx_e[('w', g)]] =  q_minus
            A_ub.append(row); b_ub.append(0.0)

        A_ub = np.array(A_ub)
        b_ub = np.array(b_ub)

        res = linprog(
            c_e,
            A_ub=A_ub, b_ub=b_ub,
            A_eq=A_eq_e, b_eq=b_eq_e,
            bounds=bounds_e,
            method="highs"
        )

        if res.success:
            eta_star = res.x[idx_e[('eta',)]]
            if eta_star > best_eta + 1e-9:
                best_eta = eta_star
                best_S = S
                best_res = res

# Guardamos solución
res_e = best_res
x_e = res_e.x
eta_star = x_e[idx_e[('eta',)]]
S_star = set(best_S)

In [6]:
def get_xe(g,i): return x_e[idx_e[('x', g, i)]]
def get_ye(g):   return x_e[idx_e[('y', g)]]
def get_we(g):   return x_e[idx_e[('w', g)]]

print("===== Resultados (e) VaR (primer cuartil, α=0.25) =====")
print(f"Solver status: {res_e.status} | message: {res_e.message}")
print(f"Z_VaR = eta* (máximo) = {eta_star:.4f}")
print(f"Prob(S*) = {sum(p_leaf[g] for g in S_star):.6f}")
print(f"S* (hojas donde se garantiza u_g ≥ eta*): {sorted(list(S_star))}")

# Tabla decisiones (nodos con decisión)
sol_x_e = pd.DataFrame([{
    "node": g,
    "stage": next(t for t,gs in stages.items() if g in gs),
    "x_activos": get_xe(g,1),
    "x_bonos": get_xe(g,2),
} for g in nodes_x])

# Tabla hojas detallada (como en (c)/(d))
rows = []
for g in leaves:
    pg = parent[g]
    x_act_parent = get_xe(pg, 1)
    x_bon_parent = get_xe(pg, 2)

    wealth_act = xi[g][0] * x_act_parent
    wealth_bon = xi[g][1] * x_bon_parent
    wealth = wealth_act + wealth_bon

    y = get_ye(g); w = get_we(g)
    u = q_plus*y - q_minus*w

    rows.append({
        "leaf": g,
        "stage": 4,
        "prob": p_leaf[g],
        "parent_node": pg,
        "x_act_parent": x_act_parent,
        "x_bon_parent": x_bon_parent,
        "wealth_activos": wealth_act,
        "wealth_bonos": wealth_bon,
        "wealth_final": wealth,
        "y_exceso": y,
        "w_deficit": w,
        "%bonos": (wealth_bon/wealth*100.0) if wealth > 1e-12 else 0.0,
        "u_g": u,
        "en_S*": g in S_star,
        "u_g == eta*": abs(u - eta_star) < 1e-6
    })

sol_leaf_e = pd.DataFrame(rows).sort_values("leaf")

display(sol_x_e.round(4))
display(sol_leaf_e.round(4))

===== Resultados (e) VaR (primer cuartil, α=0.25) =====
Solver status: 0 | message: Optimization terminated successfully. (HiGHS Status 7: Optimal)
Z_VaR = eta* (máximo) = -109.1584
Prob(S*) = 0.751315
S* (hojas donde se garantiza u_g ≥ eta*): [15]


,node,stage,x_activos,x_bonos
0,1,1,0.0,55000.0
1,2,2,0.0,62700.0
2,3,2,0.0,61600.0
3,4,3,71478.0,0.0
4,5,3,70224.0,0.0
5,6,3,70224.0,0.0
6,7,3,0.0,68992.0


,leaf,stage,prob,parent_node,x_act_parent,x_bon_parent,wealth_activos,wealth_bonos,wealth_final,y_exceso,w_deficit,%bonos,u_g,en_S*,u_g == eta*
0,8,4,0.0008,4,71478.0,0.0,89347.50,0.00,89347.50,9347.5,0.00,0.0,93.4750,False,False
1,9,4,0.0075,4,71478.0,0.0,75766.68,0.00,75766.68,0.0,4233.32,0.0,-169.3328,False,False
2,10,4,0.0075,5,70224.0,0.0,87780.00,0.00,87780.00,7780.0,0.00,0.0,77.8000,False,False
3,11,4,0.0751,5,70224.0,0.0,74437.44,0.00,74437.44,0.0,5562.56,0.0,-222.5024,False,False
4,12,4,0.0075,6,70224.0,0.0,87780.00,0.00,87780.00,7780.0,0.00,0.0,77.8000,False,False
5,13,4,0.0751,6,70224.0,0.0,74437.44,0.00,74437.44,0.0,5562.56,0.0,-222.5024,False,False
6,14,4,0.0751,7,0.0,68992.0,0.00,78650.88,78650.88,0.0,1349.12,100.0,-53.9648,False,False
7,15,4,0.7513,7,0.0,68992.0,0.00,77271.04,77271.04,0.0,2728.96,100.0,-109.1584,True,True


### Interpretación de resultados (VaR: primer cuartil)

En el modelo VaR con $\alpha=0.25$ se maximiza $\eta$, que representa el **primer cuartil (VaR al 25%)** de la utilidad final por escenario
$$
u_g = q^+y^g - q^-w^g.
$$
Es decir, se busca el mayor umbral $\eta$ tal que se cumpla $u_g \ge \eta$ en un conjunto de hojas cuya probabilidad total sea al menos $0.75$.

En este caso ($N=9$), $p=\frac{1}{11}$ y por tanto la trayectoria con tres realizaciones desfavorables tiene probabilidad
$$
(1-p)^3=\left(\frac{10}{11}\right)^3\approx 0.7513>0.75.
$$
Esto implica que para alcanzar el 75% de probabilidad basta con garantizar la cota en dicha hoja dominante. En consecuencia, el algoritmo obtiene $S^*=\{15\}$ con $\mathrm{Prob}(S^*)\approx 0.7513$.

El valor óptimo es:
$$
Z_{\text{VaR}}=\eta^*=-109.1584.
$$
Este valor queda determinado por la propia hoja 15 (la de mayor probabilidad), en la que se observa un déficit final $w^{15}=2728.9600$ (con $y^{15}=0$), de modo que
$$
u_{15}=-q^-w^{15}=-0.04\cdot 2728.9600=-109.1584=\eta^*.
$$
Por tanto, para $N=9$ el VaR del primer cuartil resulta fuertemente condicionado por el escenario de mayor probabilidad, y la política óptima se orienta a proteger dicho caso.

## (f) Calcula para cada uno de los problemas anteriores, incluido el problema estocástico neutral al riesgo, cada una de las funciones objetivo planteadas. Compara los resultados obtenidos.

## 1. Evaluación cruzada de funciones objetivo y comparación

En los apartados anteriores se han obtenido tres políticas óptimas distintas (una por criterio):
- Política $\pi_{RN}$: óptima del modelo neutral al riesgo (apartado c).
- Política $\pi_{WC}$: óptima del modelo de peor caso (apartado d).
- Política $\pi_{VaR}$: óptima del modelo VaR del primer cuartil (apartado e).

Para cada hoja final $g\in G_T$, definimos la utilidad final de una política como:
$$
u_g(\pi)=q^+y^g(\pi)-q^-w^g(\pi).
$$

A partir de estos valores, consideramos las tres funciones objetivo planteadas:

1. **Neutral al riesgo (esperanza)**:
$$
J_{RN}(\pi)=\sum_{g\in G_T} p_g\,u_g(\pi).
$$

2. **Peor escenario (minimax)**:
$$
J_{WC}(\pi)=\min_{g\in G_T} u_g(\pi).
$$

3. **VaR del primer cuartil ($\alpha=0.25$)**:
$$
J_{VaR}(\pi)=\mathrm{VaR}_{0.25}(u(\pi)),
$$
donde $\mathrm{VaR}_{0.25}$ es el mayor $\eta$ tal que $\mathbb{P}(u_g(\pi)\ge \eta)\ge 0.75$.

A continuación se calculan $J_{RN}$, $J_{WC}$ y $J_{VaR}$ para cada una de las tres políticas $\pi_{RN}$, $\pi_{WC}$ y $\pi_{VaR}$, y se comparan los resultados.

## 2. Resolución

In [7]:
# ===========
# Helpers
# ===========

def extract_u_from_solution(x_vec, idx_map, leaves, q_plus, q_minus):
    """
    Devuelve diccionario u_g = q+ y_g - q- w_g para cada hoja g.
    """
    u = {}
    for g in leaves:
        y = x_vec[idx_map[('y', g)]]
        w = x_vec[idx_map[('w', g)]]
        u[g] = q_plus*y - q_minus*w
    return u

def J_RN(u, p_leaf):
    """Esperanza: sum_g p_g u_g"""
    return sum(p_leaf[g] * u[g] for g in u.keys())

def J_WC(u):
    """Peor caso: min_g u_g"""
    return min(u.values())

def VaR_alpha(u, p_leaf, alpha=0.25):
    """
    VaR (primer cuartil si alpha=0.25) definido como:
    max eta s.t. P(u >= eta) >= 1-alpha.
    Implementación discreta con probabilidades no uniformes.
    """
    target = 1 - alpha  # 0.75
    # Ordenamos de mejor a peor (u grande a u pequeño) y acumulamos prob
    items = sorted([(u[g], p_leaf[g], g) for g in u.keys()], key=lambda t: t[0], reverse=True)
    cum = 0.0
    for val, prob, g in items:
        cum += prob
        if cum + 1e-12 >= target:
            return val
    return items[-1][0]  # fallback


# =========================
# (f) Evaluación cruzada
# =========================

alpha = 0.25

policies = {
    "(c) Política RN-opt":  (x,   idx),
    "(d) Política WC-opt":  (x_d, idx_d),
    "(e) Política VaR-opt": (x_e, idx_e),
}

rows = []
u_tables = {}

for pol_name, (x_vec, idx_map) in policies.items():
    u = extract_u_from_solution(x_vec, idx_map, leaves, q_plus, q_minus)

    jr = J_RN(u, p_leaf)
    jw = J_WC(u)
    jv = VaR_alpha(u, p_leaf, alpha=alpha)

    rows.append({
        "Política": pol_name,
        "J_RN = E[u]": jr,
        "J_WC = min(u_g)": jw,
        "J_VaR25": jv,
    })

    # Tabla de detalle por hoja (útil para comentar qué hoja ata cada criterio)
    u_tables[pol_name] = pd.DataFrame({
        "leaf": list(u.keys()),
        "prob": [p_leaf[g] for g in u.keys()],
        "u_g":  [u[g] for g in u.keys()],
    }).sort_values("leaf")

comp = pd.DataFrame(rows)
display(comp.round(4))

# (Opcional) detalle por política
for pol_name in policies.keys():
    print("\n--- u_g por hoja para", pol_name, "---")
    display(u_tables[pol_name].round(4))

,Política,J_RN = E[u],J_WC = min(u_g),J_VaR25
0,(c) Política RN-opt,-94.1521,-109.1584,-109.1584
1,(d) Política WC-opt,-103.1417,-109.1584,-109.1584
2,(e) Política VaR-opt,-119.5336,-222.5024,-109.1584



--- u_g por hoja para (c) Política RN-opt ---


,leaf,prob,u_g
0,8,0.0008,14.8492
1,9,0.0075,0.5536
2,10,0.0075,0.5536
3,11,0.0751,-53.9648
4,12,0.0075,0.5536
5,13,0.0751,-53.9648
6,14,0.0751,-53.9648
7,15,0.7513,-109.1584



--- u_g por hoja para (d) Política WC-opt ---


,leaf,prob,u_g
0,8,0.0008,121.7083
1,9,0.0075,-73.5653
2,10,0.0075,-53.9648
3,11,0.0751,-109.1584
4,12,0.0075,25.8507
5,13,0.0751,-109.1584
6,14,0.0751,-53.9648
7,15,0.7513,-109.1584



--- u_g por hoja para (e) Política VaR-opt ---


,leaf,prob,u_g
0,8,0.0008,93.4750
1,9,0.0075,-169.3328
2,10,0.0075,77.8000
3,11,0.0751,-222.5024
4,12,0.0075,77.8000
5,13,0.0751,-222.5024
6,14,0.0751,-53.9648
7,15,0.7513,-109.1584


## 3. Comparación de criterios para las políticas óptimas obtenidas

En los apartados (c)–(e) se obtuvieron tres políticas distintas: la óptima neutral al riesgo (RN), la óptima de peor caso (WC) y la óptima VaR (primer cuartil, $\alpha=0.25$). Para comparar el compromiso entre criterios, se han evaluado las tres funciones objetivo (media, peor caso y VaR25) sobre cada política, obteniéndose:

- **Política RN-opt (apartado c)**:  
  $J_{RN}=-94.1521,\quad J_{WC}=-109.1584,\quad J_{VaR25}=-109.1584.$

- **Política WC-opt (apartado d)**:  
  $J_{RN}=-103.1417,\quad J_{WC}=-109.1584,\quad J_{VaR25}=-109.1584.$

- **Política VaR-opt (apartado e)**:  
  $J_{RN}=-119.5336,\quad J_{WC}=-222.5024,\quad J_{VaR25}=-109.1584.$

Se observa que la política RN-opt maximiza el valor esperado (es la que presenta el mayor $J_{RN}$), mientras que la política WC-opt está diseñada para mejorar el comportamiento en el peor escenario, aunque aquí coincide en $J_{WC}$ con RN-opt.

Además, los valores de $J_{VaR25}$ coinciden en las tres políticas. Esto se explica porque, para $N=9$, la trayectoria desfavorable (tres realizaciones *Low*) tiene probabilidad
$$
(1-p)^3=\left(\frac{10}{11}\right)^3\approx 0.7513>0.75,
$$
de modo que el cuantil del 25% queda dominado por dicha trayectoria y el VaR25 resulta prácticamente igual para distintas políticas.

Por último, la política VaR-opt presenta un peor caso mucho más negativo ($J_{WC}=-222.5024$). Esto es coherente con el criterio VaR: al exigir únicamente que se cumpla $u_g\ge \eta$ en un subconjunto de escenarios con probabilidad total al menos $0.75$, puede “sacrificar” escenarios fuera de ese subconjunto (de probabilidad pequeña), empeorando notablemente el mínimo $u_g$ sin afectar al VaR25.

## (g) Adjunta los ficheros MPS de cada uno de los problemas. anteriores.

In [8]:
import pyomo.environ as pyo

def export_lp_to_mps(filename, c, A_eq=None, b_eq=None, A_ub=None, b_ub=None, bounds=None, model_name=None):
    c = np.asarray(c, dtype=float)
    n = len(c)

    m = pyo.ConcreteModel()
    m.I = pyo.RangeSet(0, n-1)
    m.x = pyo.Var(m.I)

    # bounds
    if bounds is None:
        bounds = [(0, None)] * n
    for i, (lb, ub) in enumerate(bounds):
        if lb is not None:
            m.x[i].setlb(float(lb))
        if ub is not None:
            m.x[i].setub(float(ub))

    # objective: minimize c^T x
    m.obj = pyo.Objective(expr=sum(float(c[i]) * m.x[i] for i in range(n)),
                          sense=pyo.minimize)

    # equalities
    m.eq = pyo.ConstraintList()
    if A_eq is not None:
        A_eq = np.asarray(A_eq, dtype=float)
        b_eq = np.asarray(b_eq, dtype=float)
        for r in range(A_eq.shape[0]):
            m.eq.add(sum(float(A_eq[r, i]) * m.x[i] for i in range(n)) == float(b_eq[r]))

    # inequalities
    m.ub = pyo.ConstraintList()
    if A_ub is not None:
        A_ub = np.asarray(A_ub, dtype=float)
        b_ub = np.asarray(b_ub, dtype=float)
        for r in range(A_ub.shape[0]):
            m.ub.add(sum(float(A_ub[r, i]) * m.x[i] for i in range(n)) <= float(b_ub[r]))

    # write MPS
    m.write(filename, io_options={"symbolic_solver_labels": True})

    # patch NAME line (optional)
    if model_name is not None:
        with open(filename, "r", encoding="utf-8", errors="ignore") as f:
            lines = f.readlines()
        for k, line in enumerate(lines):
            if line.strip().startswith("NAME"):
                lines[k] = f"NAME {model_name}\n"
                break
        with open(filename, "w", encoding="utf-8") as f:
            f.writelines(lines)

    print("Exportado:", filename, ("(NAME="+model_name+")" if model_name else ""))

In [9]:
# (e) reconstruir A_ub_e, b_ub_e SOLO con S_star óptimo
A_ub_e = []
b_ub_e = []
for g in sorted(S_star):
    row = np.zeros(n_e)
    row[idx_e[('eta',)]] = 1
    row[idx_e[('y', g)]] = -q_plus
    row[idx_e[('w', g)]] =  q_minus
    A_ub_e.append(row); b_ub_e.append(0.0)

A_ub_e = np.array(A_ub_e)
b_ub_e = np.array(b_ub_e)

# Exportar
export_lp_to_mps("modelo_c_RN.mps", c, A_eq=A_eq, b_eq=b_eq, bounds=bounds, model_name="modelo_c_RN")
export_lp_to_mps("modelo_d_WC.mps", c_d, A_eq=A_eq_d, b_eq=b_eq_d, A_ub=A_ub_d, b_ub=b_ub_d, bounds=bounds_d, model_name="modelo_d_WC")
export_lp_to_mps("modelo_e_VaR25.mps", c_e, A_eq=A_eq_e, b_eq=b_eq_e, A_ub=A_ub_e, b_ub=b_ub_e, bounds=bounds_e, model_name="modelo_e_VaR25")

Exportado: modelo_c_RN.mps (NAME=modelo_c_RN)
Exportado: modelo_d_WC.mps (NAME=modelo_d_WC)
Exportado: modelo_e_VaR25.mps (NAME=modelo_e_VaR25)


In [10]:
import os
for f in ["modelo_c_RN.mps","modelo_d_WC.mps","modelo_e_VaR25.mps"]:
    print(f, "->", os.path.exists(f))

modelo_c_RN.mps -> True
modelo_d_WC.mps -> True
modelo_e_VaR25.mps -> True
